# Estatística sobre o agente (AO1 da D3) — AMS

Parte escrita da AO1. Fica na pasta `ao1/`, com o `ao1.py` e o arquivo de casos. Kernel do `.venv` do projeto; rode de cima para baixo.
Agente: **AMS — Assistente de Manutenção Segura** (regras com limiares + estado da frota, sem LLM). Métrica primária: **acerto da decisão** = decisões iguais ao rótulo / total.
Semente de todo o notebook: **`SEMENTE = 20261126`**.

- **Conjunto:** `casos-ams-60.csv` — 60 casos rotulados (C01–C30 = a reserva da D2; C31–C60 = ampliação).
- **De onde vieram os casos novos:** do pacote de apoio da AO1 (`casos-ams-60.csv`, dados sintéticos da coordenação do curso, copiados sem alteração; ver `FONTE-AMPLIACAO.md` do pacote). **Não foram rotulados por mim caso a caso** — o AI4I 2020 não traz a ação correta rotulada, então não havia fonte real. É um limite do trabalho e fica declarado.
- **A:** linha de base da D2 (`decidir()` de `src/ams/baseline.py`). **B:** faixa de atenção na vibração — se 7,0 < vibração ≤ 7,5 mm/s e a temperatura é normal, a B manda `inspecionar_antes_de_operar` em vez de `parar_para_manutencao`; acima disso, ou com temperatura alta, age como a A. É uma regra de decisão nova (uma ação intermediária no lugar do limiar duro), não o ajuste de um limite.

> As seções "Intervalo por reamostragem", "Tamanho de amostra" e "Correlação e causalidade" são do E3/E4 e seriam entregues com a N2; foram **adiantadas** aqui.

In [1]:
%run ao1.py

60 casos
A acertou 47 · B acertou 51
discordam no C11 · A: 0 · B: 1
discordam no C22 · A: 0 · B: 1
discordam no C32 · A: 0 · B: 1
discordam no C35 · A: 1 · B: 0
discordam no C37 · A: 0 · B: 1
discordam no C42 · A: 0 · B: 1
discordam no C47 · A: 0 · B: 1
discordam no C49 · A: 1 · B: 0
latência (µs) em 3000 decisões: p50 5.3 · p95 9.6 · média 5.0 · máximo 60.3


In [2]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

AO1 = Path.cwd().resolve()           # o notebook roda de dentro de ao1/
RAIZ = AO1.parent
sys.path.insert(0, str(AO1 / "comum"))

from reamostragem import intervalo
from amostra import casos_para_diferenca, poder_para
from ams.baseline import EstadoDaFrota, carregar_casos   # decidir (A) e decidir_b (B) vêm do ao1.py

SEMENTE = 20261126          # <<< semente registrada: sem ela o número não é reproduzível
casos = pd.read_csv(AO1 / "casos-ams-60.csv", encoding="utf-8-sig")
print("casos:", len(casos), "| semente:", SEMENTE)
casos.head()

casos: 60 | semente: 20261126


,id_caso,data,tipo_maquina,idade_meses,temperatura_c,vibracao_mm_s,horas_desde_manutencao,acao_correta,falhou_depois
0,C01,2026-03-02,TORNO,72,62.0,3.2,180.0,seguir_operando,0
1,C02,2026-03-02,PRENSA,48,81.5,4.1,96.0,parar_para_manutencao,1
2,C03,2026-03-03,COMPRESSOR,24,64.0,8.4,50.0,parar_para_manutencao,1
3,C04,2026-03-03,ROBOT,36,58.0,2.1,120.0,seguir_operando,0
4,C05,2026-03-04,TORNO,66,63.0,6.8,200.0,seguir_operando,0


## As duas variantes e a latência (resumo)
**Pareamento:** A acertou 47 e B acertou 51 dos mesmos 60 casos; discordam em 8 — B acerta e A erra em 6 (C11, C22, C32, C37, C42, C47), A acerta e B erra em 2 (C35, C49). Detalhe e teste na seção 2.

**Latência (p50 e p95, e por que a média engana):** medida na seção "Latência" abaixo, com o parágrafo sobre a média.

In [3]:
# as medidas saem do medidas.py (lab 03); o relatório com as decisões escritas é o relatorio-60.md, que NÃO é sobrescrito
!python ../d3/D3-03-estatistica-descritiva/medidas.py casos-ams-60.csv --saida relatorio-60-medidas.md
from IPython.display import Markdown
Markdown(open("relatorio-60.md", encoding="utf-8").read())

conjunto: casos-ams-60.csv · 60 linhas · 9 colunas
extremos marcados: 11 · células faltantes: 0
relatório: relatorio-60-medidas.md — preencha os blocos R: e rode python teste.py


# O meu dado, com rigor — relatório do Laboratório 03

Conjunto: `casos-ams-60.csv` · 60 linhas · 9 variáveis

Medidas geradas pelo `medidas.py`: desvio padrão amostral (ddof=1),
quartis por interpolação linear, extremos pela regra da amplitude
interquartil — fora de 1,5 × a caixa (slide 24). As medidas são do
programa; as decisões são SUAS. Preencha todos os blocos `R:` e
confira com `python teste.py`.

## Medidas por variável

| variável | tipo | n | faltantes | média | mediana | desvio | Q1 | Q3 | AIQ | mín | máx |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| id_caso | categórica (60 valores; moda: C41) | 60 | 0 | — | — | — | — | — | — | — | — |
| data | categórica (30 valores; moda: 2026-03-18) | 60 | 0 | — | — | — | — | — | — | — | — |
| tipo_maquina | categórica (4 valores; moda: TORNO) | 60 | 0 | — | — | — | — | — | — | — | — |
| idade_meses | numérica | 60 | 0 | 52.1833 | 53.0000 | 28.3067 | 36.0000 | 64.0000 | 28.0000 | 3.0000 | 216.0000 |
| temperatura_c | numérica | 60 | 0 | 65.7617 | 63.0000 | 7.1741 | 61.0000 | 66.6250 | 5.6250 | 58.0000 | 83.0000 |
| vibracao_mm_s | numérica | 60 | 0 | 4.3892 | 3.6500 | 1.9195 | 3.0000 | 5.5750 | 2.5750 | 2.0000 | 8.6000 |
| horas_desde_manutencao | numérica | 60 | 0 | 118.9667 | 107.5000 | 51.7399 | 85.0000 | 152.5000 | 67.5000 | 20.0000 | 230.0000 |
| acao_correta | categórica (3 valores; moda: seguir_operando) | 60 | 0 | — | — | — | — | — | — | — | — |
| falhou_depois | numérica | 60 | 0 | 0.3333 | 0.0000 | 0.4754 | 0.0000 | 1.0000 | 1.0000 | 0.0000 | 1.0000 |

## Extremos marcados (fora de 1,5 × a caixa — slide 24)

### Extremos de `idade_meses` — 1 valor(es), limites [-6.0000; 106.0000]

| linha | valor |
|---:|---:|
| 12 | 216.0000 |

Decisão sobre ESTES extremos — manter ou remover, caso a caso,
com o motivo (slide 25: quem remove sem dizer, mente por omissão):

R: **Manter** (1 valor: C12, 216 meses). É o único candidato a erro de digitação (o seguinte é 80 meses), mas 18 anos é plausível para uma prensa e a regra de decisão do agente **não usa `idade_meses`**, então o valor não afeta nenhuma métrica da AO1. Remover sem evidência de erro seria omissão (slide 25). Pendência: confirmar no registro da máquina antes de qualquer análise que use idade.

### Extremos de `temperatura_c` — 10 valor(es), limites [52.5625; 75.0625]

| linha | valor |
|---:|---:|
| 2 | 81.5000 |
| 6 | 77.5000 |
| 13 | 79.2000 |
| 17 | 76.0000 |
| 21 | 80.0000 |
| 27 | 82.5000 |
| 34 | 79.5000 |
| 41 | 83.0000 |
| 43 | 79.0000 |
| 53 | 81.0000 |

Decisão sobre ESTES extremos — manter ou remover, caso a caso,
com o motivo (slide 25: quem remove sem dizer, mente por omissão):

R: **Manter os 10.** Oito têm rótulo `parar_para_manutencao` (temperatura acima de 78 °C é o motivo da parada) e dois (C06 = 77,5 °C e C17 = 76,0 °C) são `seguir_operando`, controles logo abaixo do limite da regra. São sinal, não erro: removê-los tiraria do conjunto os casos que testam a regra de temperatura. Sem eles (50 casos) o acerto da linha de base iria de 0,783 para 0,740: o número mudaria, mas porque o conjunto deixaria de ter o fenômeno, não porque ficou mais verdadeiro.

## Faltantes (os três mecanismos do slide 26)

Nenhuma variável tem valores faltantes.

## As quatro perguntas do slide 29

**1. Média e mediana se separam em alguma variável? Em qual?**

R: Sim, em `vibracao_mm_s`: média 4,39 contra mediana 3,65 (a média é ~20% maior), por causa da cauda de 14 casos entre 6,8 e 8,6 mm/s. `horas_desde_manutencao` (119,0 contra 107,5) e `temperatura_c` (65,8 contra 63,0) também puxam para cima, pelos extremos de temperatura. `idade_meses` (52,2 contra 53,0) não se separa, mesmo com o valor de 216: a média sozinha não denunciaria esse extremo. Em `vibracao_mm_s` o 1,5×AIQ não marca nada (limite superior 9,44 mm/s, máximo 8,6): a cauda é real, não aberrante.

**2. Quantos extremos foram marcados, e quantos você decidiu manter?**

R: 11 marcados (1 em `idade_meses`, 10 em `temperatura_c`; nenhum em vibração). **Mantive os 11**, pelos motivos acima. Regra adotada: só removo com evidência de erro de medição (sensor travado, unidade errada, valor impossível); aqui não há.

**3. Algum faltante parece do terceiro mecanismo?**

R: Não há faltantes: 0 células vazias nas 9 variáveis. Isso decorre de os casos serem sintéticos (coordenação do curso), não de o sensor real nunca falhar. Em produção, a hipótese que mais preocupa é o terceiro mecanismo (MNAR): um sensor que satura ou desliga justamente com vibração muito alta perderia o dado no caso crítico. Nessa hipótese descartar ou imputar pela média esconderia o alerta; a regra deveria tratar "sem leitura de vibração" como `inspecionar_antes_de_operar`, nunca como `seguir_operando`.

**4. O que a análise da D1 não tinha visto?**

R: **1.** Que a distribuição de vibração tem dois blocos separados por um vazio (5,8 a 6,8 mm/s) e que o limiar de 7,0 mm/s da regra cai dentro da cauda, não ao lado dela: 9 dos 60 casos ficam entre 7,0 e 7,6 mm/s, justamente onde o rótulo correto muda entre parar e inspecionar. **2.** Que o conjunto é desbalanceado em favor de `seguir_operando` (33 de 60, 55%): sempre responder "seguir" já daria 55%, esse é o piso a bater. **3.** Que a taxa de acerto só é interpretável com intervalo: 47/60 = 0,783, IC95% [0,683; 0,883] (notebook).

## Comparação com a análise exploratória da D1

O que mudou de lá para cá — o que a análise de lá não tinha visto,
e o que estas decisões escritas mudam no conjunto que a D5 vai usar:

R: A D1 explorou o AI4I (taxa de falha por tipo, desgaste da ferramenta) e concluiu que a falha é rara (~3-4%) e que acurácia alta ali é fácil e enganosa. Este conjunto de 60 casos é outro objeto: rotula a *decisão correta* e não só o desfecho, e tem 33% de casos com `falhou_depois` = 1, bem acima dos 3,4% do AI4I. Isso muda o que a D5 pode usar: a taxa de base de falha **não** é a do AI4I, e qualquer modelo treinado nele precisa de um conjunto de decisão próprio. As decisões escritas aqui (manter os extremos, não há faltantes, vibração como cauda real) são o que a D5 herda: nenhum extremo foi removido, então o conjunto segue com os 60 casos.


## 1. AED rigorosa do conjunto ampliado

In [4]:
print(casos.dtypes.to_string())
print()
print("rótulos:\n", casos["acao_correta"].value_counts().to_string())
print("\nfamílias:\n", casos["tipo_maquina"].value_counts().to_string())
print("\nfalhou_depois:", casos["falhou_depois"].value_counts().to_dict())
casos[["idade_meses","temperatura_c","vibracao_mm_s","horas_desde_manutencao"]].describe().round(2)

id_caso                       str
data                          str
tipo_maquina                  str
idade_meses                 int64
temperatura_c             float64
vibracao_mm_s             float64
horas_desde_manutencao    float64
acao_correta                  str
falhou_depois               int64

rótulos:
 acao_correta
seguir_operando                33
parar_para_manutencao          14
inspecionar_antes_de_operar    13

famílias:
 tipo_maquina
TORNO         16
PRENSA        16
COMPRESSOR    14
ROBOT         14

falhou_depois: {0: 40, 1: 20}


,idade_meses,temperatura_c,vibracao_mm_s,horas_desde_manutencao
count,60.00,60.00,60.00,60.00
mean,52.18,65.76,4.39,118.97
std,28.31,7.17,1.92,51.74
min,3.00,58.00,2.00,20.00
25%,36.00,61.00,3.00,85.00
50%,53.00,63.00,3.65,107.50
75%,64.00,66.62,5.58,152.50
max,216.00,83.00,8.60,230.00


In [5]:
cols = ["idade_meses","temperatura_c","vibracao_mm_s","horas_desde_manutencao"]
fig, eixos = plt.subplots(2, 4, figsize=(15, 6))
for i, c in enumerate(cols):
    eixos[0, i].hist(casos[c], bins=15, color="#4C72B0"); eixos[0, i].set_title(c)
    eixos[1, i].boxplot(casos[c], orientation="horizontal"); eixos[1, i].set_yticks([])
plt.tight_layout(); plt.savefig(AO1 / "aed-distribuicoes.png", dpi=110); plt.show()

C:\Users\Pc\AppData\Local\Temp\ipykernel_11076\3952302797.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(AO1 / "aed-distribuicoes.png", dpi=110); plt.show()


### Distribuições — leitura
- **`vibracao_mm_s`** tem dois blocos: 46 casos entre 2,0 e 5,8 mm/s (operação normal) e uma cauda isolada de 14 casos entre 6,8 e 8,6 mm/s,
  com um vazio entre 5,8 e 6,8. A cauda não é ruído de medição: é onde estão as máquinas que o agente deve parar ou inspecionar.
- **`temperatura_c`** tem a metade central entre 61 e 66,6 °C e uma cauda de 10 casos acima de 75 °C (76–83) — as paradas por temperatura (e dois controles abaixo do limite de 78).
- **`horas_desde_manutencao`** e **`idade_meses`** são largas e sem estrutura marcada; a regra da linha de base não as usa. `idade_meses` tem um valor isolado (C12 = 216 meses; o seguinte é 80).
- O conjunto é **desbalanceado em favor de `seguir_operando`** (≈55%): um agente que sempre respondesse "seguir" já acertaria ~55%. É o piso a bater.

### Outliers — critério (1,5×IQR) e decisão

In [6]:
def limites_iqr(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return q1 - 1.5*i, q3 + 1.5*i

linhas = []
for c in cols:
    lo, hi = limites_iqr(casos[c])
    fora = casos[(casos[c] < lo) | (casos[c] > hi)]
    linhas.append({"variavel": c, "limite_inf": round(lo,2), "limite_sup": round(hi,2),
                   "n_outliers": len(fora), "casos": ", ".join(fora["id_caso"])})
display(pd.DataFrame(linhas))

lo, hi = limites_iqr(casos["vibracao_mm_s"])
fora_vib = casos[(casos["vibracao_mm_s"] > hi)]
print(f"\nvibração: limite superior do IQR = {hi:.2f} mm/s, máximo observado = {casos['vibracao_mm_s'].max()} -> {len(fora_vib)} outliers por este critério")
lt, ht = limites_iqr(casos["temperatura_c"])
fora_t = casos[casos["temperatura_c"] > ht]
print("\noutliers de temperatura e o rótulo correto deles:"); print(fora_t[["id_caso","temperatura_c","acao_correta"]].to_string(index=False))
print("\nC12 (idade_meses fora do IQR):"); print(casos[casos.id_caso=="C12"][["id_caso","tipo_maquina","idade_meses","acao_correta"]].to_string(index=False))

,variavel,limite_inf,limite_sup,n_outliers,casos
0,idade_meses,-6.00,106.00,1,C12
1,temperatura_c,52.56,75.06,10,"C02, C06, C13, C17, C21, C27, C34, C41, C43, C53"
2,vibracao_mm_s,-0.86,9.44,0,
3,horas_desde_manutencao,-16.25,253.75,0,



vibração: limite superior do IQR = 9.44 mm/s, máximo observado = 8.6 -> 0 outliers por este critério

outliers de temperatura e o rótulo correto deles:
id_caso  temperatura_c          acao_correta
    C02           81.5 parar_para_manutencao
    C06           77.5       seguir_operando
    C13           79.2 parar_para_manutencao
    C17           76.0       seguir_operando
    C21           80.0 parar_para_manutencao
    C27           82.5 parar_para_manutencao
    C34           79.5 parar_para_manutencao
    C41           83.0 parar_para_manutencao
    C43           79.0 parar_para_manutencao
    C53           81.0 parar_para_manutencao

C12 (idade_meses fora do IQR):
id_caso tipo_maquina  idade_meses    acao_correta
    C12       PRENSA          216 seguir_operando


In [7]:
# O que mudaria se eu REMOVESSE os outliers de vibração e temperatura?
def acerto(decidir, df):
    estado = EstadoDaFrota(); ok = 0
    for reg in carregar_casos(AO1 / "casos-ams-60.csv"):
        if reg["id_caso"] in set(df["id_caso"]):
            ok += decidir(reg, estado) == reg["acao_correta"]
        # o estado vê TODOS os casos, como no agente real (o desfecho se revela de qualquer forma)
        estado.registrar_desfecho(reg)
    return ok, len(df)

lv, hv = limites_iqr(casos["vibracao_mm_s"]); lt, ht = limites_iqr(casos["temperatura_c"])
sem_out = casos[(casos["vibracao_mm_s"].between(lv, hv)) & (casos["temperatura_c"].between(lt, ht))]
for nome, f in [("A", decidir), ("B", decidir_b)]:
    ok, n = acerto(f, casos); ok2, n2 = acerto(f, sem_out)
    print(f"variante {nome}: com todos {ok}/{n} = {ok/n:.3f} | sem outliers {ok2}/{n2} = {ok2/n2:.3f}")

variante A: com todos 47/60 = 0.783 | sem outliers 37/50 = 0.740
variante B: com todos 51/60 = 0.850 | sem outliers 41/50 = 0.820


### Decisão sobre os outliers: **NÃO remover nenhum**
- **Vibração:** pelo critério 1,5×IQR não há nenhum outlier (limite superior 9,44 mm/s; máximo 8,6). A cauda de 6,8–8,6 mm/s é uma **cauda real, não valor aberrante** — e é
  justamente onde as variantes A e B discordam.
- **Temperatura:** 10 casos acima do limite do IQR (≈75 °C). Oito têm rótulo `parar_para_manutencao` (temperatura acima de 78 °C é o motivo da parada) e dois (C06 = 77,5 °C e
  C17 = 76,0 °C) são `seguir_operando` — controles logo abaixo do limite da regra. São **sinal, não erro**: removê-los tira do conjunto os casos que testam a regra de temperatura.
- **O que mudaria se removesse** (célula acima): o conjunto cai para 50 casos e o acerto de A vai de 0,783 para 0,740 e o de B de 0,850 para 0,820. A diferença B−A fica em 8 pontos
  (4 casos de 50 contra 4 de 60), mas sobre um conjunto sem a cauda do fenômeno — número menos representativo, não mais verdadeiro.
- **Idade:** C12 (216 meses, 18 anos) é o único valor candidato a erro de digitação (o seguinte é 80). É fisicamente plausível e **a regra de decisão não usa `idade_meses`**,
  então não afeta nenhuma métrica deste trabalho. Mantido; a pendência é confirmar o valor no registro da máquina antes de qualquer análise que use idade.
- Regra geral adotada: só removo um valor com evidência de erro de medição (sensor travado, unidade errada, valor impossível). Aqui não há.

### Faltantes — hipótese de mecanismo

In [8]:
print("faltantes por coluna nos 60 casos:"); print(casos.isna().sum().to_string())
ai4i_raw = RAIZ / "data" / "raw" / "ai4i2020.csv"          # completo (10.000), baixado por scripts/baixar-dados.py
if not ai4i_raw.exists():
    ai4i_raw = RAIZ / "data" / "amostra" / "ai4i2020.csv"   # amostra versionada (200 linhas): os números mudam!
print("arquivo usado:", ai4i_raw)
ai4i = pd.read_csv(ai4i_raw)
print("\nAI4I:", ai4i_raw.name, ai4i.shape, "| faltantes no total:", int(ai4i.isna().sum().sum()))

faltantes por coluna nos 60 casos:
id_caso                   0
data                      0
tipo_maquina              0
idade_meses               0
temperatura_c             0
vibracao_mm_s             0
horas_desde_manutencao    0
acao_correta              0
falhou_depois             0
arquivo usado: C:\CEDAGI\Projeto-AMS\data\raw\ai4i2020.csv

AI4I: ai4i2020.csv (10000, 14) | faltantes no total: 0


### Faltantes — leitura
**Não há valores faltantes** nos 60 casos nem no AI4I. Isso é consequência de como os dados foram feitos (sintéticos / simulados),
não prova de que o sensor real nunca falha. Por isso a hipótese de mecanismo é declarada para o agente em produção:

| Hipótese | No AMS, seria… | O que autoriza |
|---|---|---|
| **MCAR** (completamente ao acaso) | sensor que perde pacotes por falha de rede, sem relação com a leitura | descartar as linhas incompletas só custa poder, não enviesa |
| **MAR** (ao acaso dado o observado) | vibração ausente mais vezes em um `tipo_maquina` (ex.: ROBOT sem sensor instalado) | imputar/modelar condicionando em `tipo_maquina` |
| **MNAR** (depende do valor ausente) | sensor que **satura/desliga justamente com vibração muito alta** | NÃO descartar nem imputar pela média: o faltante *é* o alerta. A regra do agente deveria tratar "sem leitura de vibração" como `inspecionar`, nunca como `seguir_operando` |

A hipótese que mais me preocupa é **MNAR**: é a única em que ignorar o faltante faz o agente falhar justamente no caso crítico.

## 2. As duas variantes sobre os MESMOS 60 casos

In [9]:
def avaliar(decidir_fn):
    estado = EstadoDaFrota(); linhas = []
    for reg in carregar_casos(AO1 / "casos-ams-60.csv"):
        dec = decidir_fn(reg, estado)
        linhas.append({"id_caso": reg["id_caso"], "decisao": dec, "correta": reg["acao_correta"], "acertou": int(dec == reg["acao_correta"])})
        estado.registrar_desfecho(reg)
    return linhas

resA = pd.DataFrame(avaliar(decidir)).rename(columns={"decisao":"dec_a","acertou":"acerto_a"})
resB = pd.DataFrame(avaliar(decidir_b)).rename(columns={"decisao":"dec_b","acertou":"acerto_b"})
res = resA.merge(resB[["id_caso","dec_b","acerto_b"]], on="id_caso")
acertos = pd.read_csv(AO1 / "acertos.csv")
assert list(acertos["linha_de_base"]) == list(res["acerto_a"]) and list(acertos["variante_b"]) == list(res["acerto_b"]), "acertos.csv fora de sincronia"
assert len(acertos) == 60 and set(np.unique(acertos.values)) <= {0, 1}

print(f"A (linha de base): {acertos.linha_de_base.sum()}/60 = {acertos.linha_de_base.mean():.4f}")
print(f"B (faixa de atenção): {acertos.variante_b.sum()}/60 = {acertos.variante_b.mean():.4f}")
print("\nTabela de pareamento (linhas = A, colunas = B):")
tab = pd.crosstab(acertos.linha_de_base, acertos.variante_b); display(tab)
so_a = int(((acertos.linha_de_base==1)&(acertos.variante_b==0)).sum()); so_b = int(((acertos.linha_de_base==0)&(acertos.variante_b==1)).sum())
print(f"discordantes: só A acerta = {so_a} | só B acerta = {so_b}")
print("\nCasos em que as variantes discordam:")
display(res[res.acerto_a != res.acerto_b][["id_caso","correta","dec_a","dec_b"]])

A (linha de base): 47/60 = 0.7833
B (faixa de atenção): 51/60 = 0.8500

Tabela de pareamento (linhas = A, colunas = B):


variante_b,0,1
linha_de_base,,
0,7,6
1,2,45


discordantes: só A acerta = 2 | só B acerta = 6

Casos em que as variantes discordam:


,id_caso,correta,dec_a,dec_b
10,C11,inspecionar_antes_de_operar,parar_para_manutencao,inspecionar_antes_de_operar
21,C22,inspecionar_antes_de_operar,parar_para_manutencao,inspecionar_antes_de_operar
31,C32,inspecionar_antes_de_operar,parar_para_manutencao,inspecionar_antes_de_operar
34,C35,parar_para_manutencao,parar_para_manutencao,inspecionar_antes_de_operar
36,C37,inspecionar_antes_de_operar,parar_para_manutencao,inspecionar_antes_de_operar
41,C42,inspecionar_antes_de_operar,parar_para_manutencao,inspecionar_antes_de_operar
46,C47,inspecionar_antes_de_operar,parar_para_manutencao,inspecionar_antes_de_operar
48,C49,parar_para_manutencao,parar_para_manutencao,inspecionar_antes_de_operar


**A variante B muda comportamento, não parâmetro:** introduz uma faixa de atenção (7,0 < vibração ≤ 7,5 mm/s, temperatura normal → `inspecionar`)
no lugar do limiar duro "parar". É uma regra de decisão nova, motivada pela análise de erros da D2 (falsos positivos C11 e C22).

**Pareamento:** as duas variantes avaliam os mesmos 60 casos, então o que importa são os **discordantes** (a diferença aparece só neles).
Só os pares discordantes carregam informação sobre qual variante é melhor — é por isso que o pareamento compra poder (calculado na seção 4).

**Limite que precisa ficar escrito:** a faixa foi desenhada olhando os erros deste mesmo conjunto. A vantagem de B aqui está inflada por esse
reuso; num conjunto novo ela provavelmente encolhe. Ganho medido no mesmo dado que inspirou a regra não vale como evidência independente.

### Intervalo de confiança por reamostragem (`comum/reamostragem.py`, semente registrada)

In [10]:
iA = intervalo(acertos.linha_de_base.values, semente=SEMENTE)
iB = intervalo(acertos.variante_b.values, semente=SEMENTE)
# diferença pareada: reamostra os casos (a coluna d = b - a mantém o pareamento)
d = (acertos.variante_b - acertos.linha_de_base).values
iD = intervalo(d, semente=SEMENTE)

print(f"semente = {iA.semente} | repetições = {iA.repeticoes} | nível = {iA.nivel}")
for nome, i in [("A", iA), ("B", iB)]:
    print(f"variante {nome}: acerto = {i.estimativa:.4f}  IC95% = [{i.inferior:.4f}; {i.superior:.4f}]  (largura {i.largura:.3f})")
print(f"diferença pareada B−A = {iD.estimativa:+.4f}  IC95% = [{iD.inferior:+.4f}; {iD.superior:+.4f}]")
print("o IC da diferença contém zero?", iD.inferior <= 0 <= iD.superior)

fig, ax = plt.subplots(figsize=(7, 2.6))
for y, (nome, i) in enumerate([("B", iB), ("A", iA)]):
    ax.errorbar(i.estimativa, y, xerr=[[i.estimativa-i.inferior],[i.superior-i.estimativa]], fmt="o", capsize=5)
ax.set_yticks([0,1]); ax.set_yticklabels(["B (faixa)","A (base)"]); ax.set_xlabel("acerto (IC 95%)")
plt.tight_layout(); plt.savefig(AO1 / "ic-variantes.png", dpi=110); plt.show()

semente = 20261126 | repetições = 10000 | nível = 0.95
variante A: acerto = 0.7833  IC95% = [0.6833; 0.8833]  (largura 0.200)
variante B: acerto = 0.8500  IC95% = [0.7500; 0.9333]  (largura 0.183)
diferença pareada B−A = +0.0667  IC95% = [-0.0167; +0.1667]
o IC da diferença contém zero? True


C:\Users\Pc\AppData\Local\Temp\ipykernel_11076\591855871.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(AO1 / "ic-variantes.png", dpi=110); plt.show()


### Leitura (a resposta honesta é "não sei ainda")
B acerta 4 casos a mais que A (+6,7 pontos), mas **os dois intervalos se sobrepõem bastante** e o intervalo da diferença pareada **contém zero**.
Com 6 casos em que só B acerta e 2 em que só A acerta, um teste exato (McNemar) dá p ≈ 0,29 — veja a célula abaixo. Com 60 casos, essa magnitude é
indistinguível de acaso. **Não posso afirmar que B é melhor que A**; posso afirmar que B foi melhor *neste conjunto*, e que esse conjunto inspirou a regra.

In [11]:
from math import comb
n = so_a + so_b; k = min(so_a, so_b)
p_exato = min(1.0, 2 * sum(comb(n, i) for i in range(0, k + 1)) / 2**n)
print(f"McNemar exato (bilateral): discordantes = {n}  (só A = {so_a}, só B = {so_b})  p = {p_exato:.4f}")

McNemar exato (bilateral): discordantes = 8  (só A = 2, só B = 6)  p = 0.2891


### Latência — p50 e p95 sobre as invocações do agente

In [12]:
# O AMS da D2 é um agente de regras (sem LLM): a latência é a de `decidir` por invocação.
# Medimos cada invocação separadamente (perf_counter_ns), em ordem cronológica, repetindo o conjunto 200x
# (12.000 invocações) para ter distribuição; a primeira rodada é descartada como aquecimento.
registros = carregar_casos(AO1 / "casos-ams-60.csv")
def medir_latencias(decidir, repeticoes=200):
    tempos = []
    for r in range(repeticoes + 1):
        estado = EstadoDaFrota()
        for reg in registros:
            t0 = time.perf_counter_ns(); decidir(reg, estado); t1 = time.perf_counter_ns()
            if r > 0: tempos.append((t1 - t0) / 1000)   # microssegundos
            estado.registrar_desfecho(reg)
    return np.array(tempos)

lat = {"A": medir_latencias(decidir), "B": medir_latencias(decidir_b)}
for nome, t in lat.items():
    print(f"variante {nome}: n={len(t)}  média={t.mean():.2f} µs  p50={np.percentile(t,50):.2f} µs  p95={np.percentile(t,95):.2f} µs  máx={t.max():.1f} µs")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(lat["A"], bins=np.linspace(0, np.percentile(lat["A"], 99.5), 80), color="#4C72B0")
ax.axvline(lat["A"].mean(), color="r", ls="--", label="média"); ax.axvline(np.percentile(lat["A"],50), color="g", label="p50"); ax.axvline(np.percentile(lat["A"],95), color="k", label="p95")
ax.set_xlabel("latência por invocação (µs)"); ax.legend(); plt.tight_layout(); plt.savefig(AO1 / "latencia.png", dpi=110); plt.show()
ic_p95 = intervalo(lat["A"], estatistica=lambda x: np.percentile(x, 95), repeticoes=2000, semente=SEMENTE)
print(f"IC95% do p95 (variante A, 2000 reamostragens, semente {SEMENTE}): [{ic_p95.inferior:.2f}; {ic_p95.superior:.2f}] µs")

variante A: n=12000  média=3.49 µs  p50=3.40 µs  p95=7.90 µs  máx=309.0 µs
variante B: n=12000  média=5.88 µs  p50=5.70 µs  p95=9.70 µs  máx=3161.8 µs


C:\Users\Pc\AppData\Local\Temp\ipykernel_11076\4227162947.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_xlabel("latência por invocação (µs)"); ax.legend(); plt.tight_layout(); plt.savefig(AO1 / "latencia.png", dpi=110); plt.show()


IC95% do p95 (variante A, 2000 reamostragens, semente 20261126): [7.80; 8.00] µs


### Por que a média é enganosa em tempo de resposta
Tempo de resposta tem **cauda longa à direita** e é limitado por zero: a maior parte das invocações é rápida e algumas poucas são muito mais lentas
(coleta de lixo, troca de contexto do sistema operacional, cache frio, e — num agente com LLM — fila, rede e geração de tokens). A média é puxada
por essas poucas invocações e **não descreve nenhuma invocação real**: fica acima do que a maioria vive e abaixo do que o pior caso vive. Nas medições acima
(`ao1.py` e a célula própria, ambos sobre as invocações de `decidir`) o p95 fica em torno de 2 a 3 vezes o p50 e o máximo passa de 50 vezes o p50 — a média, pouco acima do p50, **não revela** nenhum dos dois: não diz que 1 invocação em cada 20 é bem mais lenta que a típica, nem que a pior foi dezenas de vezes pior. O **p50** diz o que o usuário típico sente; o **p95** (e p99) diz o que quem tem azar sente — é o que define o limite de um SLA,
porque o usuário se lembra da lentidão, não da média. Também não se deve ler diferença entre A e B nestes números: diferenças de décimos de µs entre A e B são ruído do sistema operacional.

*Limite desta medição:* o AMS da D2 é um agente de regras sem rede nem LLM, por isso a latência está na casa de microssegundos e é dominada por ruído do
sistema operacional. A lição vale integralmente para o agente com LLM da D10, onde a cauda será de segundos.

## 3. Tamanho de amostra e poder (`comum/amostra.py`)

In [13]:
TAXA_BASE = float(acertos.linha_de_base.mean())     # taxa de acerto do MEU agente (variante A): 47/60
print(f"taxa de base declarada (acerto da variante A): {TAXA_BASE:.4f}")
dim = casos_para_diferenca(0.05, TAXA_BASE, poder=0.80, significancia=0.05)
print(dim)
print(f"\nPara detectar +5 pontos (poder 80%, alfa 5%, correlação {dim.correlacao}):")
print(f"  desenho PAREADO (mesmos casos nas duas versões): {dim.casos_pareados} casos")
print(f"  desenho NÃO PAREADO: {dim.casos_independentes} casos por versão = {dim.avaliacoes_independentes} avaliações")
print(f"  economia do pareamento: {dim.economia_do_pareamento:.0%} das invocações")
phi = float(np.corrcoef(acertos.linha_de_base, acertos.variante_b)[0, 1])
dim_obs = casos_para_diferenca(0.05, TAXA_BASE, correlacao=round(phi, 2))
print(f"  correlação OBSERVADA entre os acertos de A e B nestes 60 casos: {phi:.2f} -> pareado exigiria {dim_obs.casos_pareados} casos (em vez de {dim.casos_pareados} com r=0,70)")
print(f"  discordantes esperados no pareado: {dim.discordantes_esperados:.1f}")

print("\nO que os 60 casos compram (poder, desenho pareado, correlação 0,70):")
linhas = [{"melhoria (pp)": int(d*100), "poder com 60 casos": round(poder_para(60, d, TAXA_BASE), 3),
           "casos p/ 80% (pareado)": casos_para_diferenca(d, TAXA_BASE).casos_pareados,
           "casos p/ 80% (não pareado, por versão)": casos_para_diferenca(d, TAXA_BASE).casos_independentes}
          for d in (0.03, 0.05, 0.08, 0.10, 0.12, 0.15)]
display(pd.DataFrame(linhas))

taxa de base declarada (acerto da variante A): 0.7833
Dimensionamento(diferenca=0.05, taxa_base=0.7833333333333333, poder=0.8, significancia=0.05, correlacao=0.7, casos_independentes=972, casos_pareados=300, discordantes_esperados=28.849269841301115)

Para detectar +5 pontos (poder 80%, alfa 5%, correlação 0.7):
  desenho PAREADO (mesmos casos nas duas versões): 300 casos
  desenho NÃO PAREADO: 972 casos por versão = 1944 avaliações
  economia do pareamento: 85% das invocações
  correlação OBSERVADA entre os acertos de A e B nestes 60 casos: 0.57 -> pareado exigiria 425 casos (em vez de 300 com r=0,70)
  discordantes esperados no pareado: 28.8

O que os 60 casos compram (poder, desenho pareado, correlação 0,70):


,melhoria (pp),poder com 60 casos,casos p/ 80% (pareado),"casos p/ 80% (não pareado, por versão)"
0,3,0.111,850,2807
1,5,0.236,300,972
2,8,0.517,116,356
3,10,0.697,77,217
4,12,0.780,63,143
5,15,0.870,50,84


### Implicação para o AMS
- Para detectar **+5 pp** com 80% de poder e 5% de significância, partindo de 78,3% de acerto (47/60), são necessários **300 casos no desenho pareado** (correlação 0,70 do kit) e
  **972 casos por versão (1.944 avaliações) no não pareado**. O pareamento economiza cerca de 85% das invocações — mas 300 é 5× os 60 casos que tenho.
- A correlação **observada** entre os acertos de A e B é só ≈ 0,57 (menor que a 0,70 assumida): isso gera mais discordantes e o pareado sobe para **≈ 425 casos**. O número honesto é, portanto,
  "entre 300 e 425 casos", não 300.
- Com **60 casos**, o poder para +5 pp é de apenas **≈ 24%**: é mais provável não ver o ganho do que vê-lo. Só diferenças de **≈ 10–12 pp** chegam a 70–78% de poder
  (10 pp: 70%; 12 pp: 78%). A faixa de atenção (B) deu +6,7 pp: **abaixo do que o conjunto consegue distinguir**, coerente com o p ≈ 0,29 e com o IC da diferença cruzando zero.
- **O que posso honestamente procurar hoje:** mudanças de ~10 pontos ou mais (trocar a regra de decisão por outra muito diferente, não afinar um limiar).
  Para ganhos de 3–5 pp, a resposta correta com este conjunto é "não sei ainda".
- **O que faria valer a pena ampliar:** (1) rotular ≈ 300 casos, com ênfase na **região de fronteira** (vibração 6,5–8 mm/s e falhas intermitentes), que é onde as
  variantes discordam — casos fáceis aumentam o n sem aumentar os discordantes; (2) um conjunto **separado** (teste) do que inspirou a regra, para B ser avaliado sem
  o viés do mesmo dado; (3) só vale ampliar se a decisão de negócio depender de ganhos pequenos; se só interessar ganho ≥ 10 pp, os 60 casos já bastam.
- *Sensibilidade:* a conta usa a correlação como parâmetro. Mudar r de 0,70 para 0,57 sobe o pareado de 300 para ~425 casos; r = 0,50 daria ~493.

## 4. Correlação e causalidade no meu domínio

In [14]:
v = ai4i[["Rotational speed [rpm]", "Torque [Nm]"]]
r = v.corr().iloc[0, 1]
print(f"AI4I completo ({len(ai4i)} registros): correlação rotação × torque = {r:.3f}")
ic_r = intervalo(np.arange(len(ai4i)), estatistica=lambda idx: np.corrcoef(v.values[idx.astype(int)].T)[0,1], repeticoes=1000, semente=SEMENTE)
print(f"IC95% (reamostragem, semente {SEMENTE}): [{ic_r.inferior:.3f}; {ic_r.superior:.3f}]")
print("\npor tipo de produto:"); print(ai4i.groupby("Type")[["Rotational speed [rpm]","Torque [Nm]"]].apply(lambda g: g.corr().iloc[0,1]).round(3).to_string())
print("\ntaxa de falha por faixa de torque:"); print(ai4i.groupby(pd.qcut(ai4i["Torque [Nm]"], 5))["Machine failure"].mean().round(3).to_string())
fig, ax = plt.subplots(figsize=(6, 4)); ax.scatter(v.iloc[:,0], v.iloc[:,1], s=3, alpha=.4)
ax.set_xlabel("rotação (rpm)"); ax.set_ylabel("torque (Nm)"); plt.tight_layout(); plt.savefig(AO1 / "correlacao-rotacao-torque.png", dpi=110); plt.show()

AI4I completo (10000 registros): correlação rotação × torque = -0.875


IC95% (reamostragem, semente 20261126): [-0.879; -0.870]

por tipo de produto:
Type
H   -0.878
L   -0.873
M   -0.877

taxa de falha por faixa de torque:
Torque [Nm]
(3.799, 31.4]    0.023
(31.4, 37.5]     0.004
(37.5, 42.7]     0.006
(42.7, 48.3]     0.022
(48.3, 76.6]     0.115


C:\Users\Pc\AppData\Local\Temp\ipykernel_11076\2720797689.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_xlabel("rotação (rpm)"); ax.set_ylabel("torque (Nm)"); plt.tight_layout(); plt.savefig(AO1 / "correlacao-rotacao-torque.png", dpi=110); plt.show()


Texto completo da página: [`correlacao-e-causalidade.md`](correlacao-e-causalidade.md). Tamanho de amostra: [`tamanho-de-amostra.md`](tamanho-de-amostra.md).